# Notebook 05: Model Evaluation, Pipelines, and Model Deployment

In this final notebook, we tie everything together into a clean, reproducible, and deployable machine learning workflow:
1. **Cross-Validation**: Comparing 5-Fold KFold and StratifiedKFold, explaining why a single train/test split is insufficient.
2. **Handling Imbalanced Data**: Comparing normal Logistic Regression, `class_weight='balanced'`, and random oversampling with `sklearn.utils.resample` on training data.
3. **End-to-End Scikit-Learn Pipelines**: Using `ColumnTransformer` with `SimpleImputer`, `StandardScaler`, and `OneHotEncoder` directly on the **RAW dataset** (`students.csv`), so the pipeline autonomously handles missing values and encoding.
4. **Model Selection and Tuning**: Comparing Logistic Regression vs KNN with cross-validation and performing a concise `GridSearchCV`.
5. **Model Serialization**: Saving the final fitted pipeline with `joblib` to `models/best_model.joblib`, and reloading it to predict a new student's result.

### 1. Importing Libraries
We import NumPy, Pandas, Scikit-Learn preprocessing, pipeline, model selection, evaluation metrics, and Joblib for model saving.

In [1]:
# Import required libraries
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Set plot style
sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

Libraries imported successfully.


### 2. Loading the RAW Dataset
Notice that we load `students.csv` (the RAW dataset with missing values and raw string categories), NOT the cleaned CSV.
This allows our Scikit-Learn Pipeline to handle missing value imputation, scaling, and one-hot encoding internally.

In [2]:
# Load raw student data
df_raw = pd.read_csv('../data/students.csv')

# Drop duplicate rows if present to start with clean unique records
df_raw = df_raw.drop_duplicates().reset_index(drop=True)

# Define feature columns and target
num_features = ['study_hours', 'attendance', 'previous_marks', 'sleep_hours']
cat_features = ['gender', 'parental_education', 'extracurricular']
all_features = num_features + cat_features

X = df_raw[all_features]
y = df_raw['result']  # Target is 'Pass' or 'Fail'

print('Dataset shape:', X.shape)
print('Missing values in raw features:')
print(X.isnull().sum())
print('\nClass distribution:')
print(y.value_counts(normalize=True).round(3))

Dataset shape: (490, 7)
Missing values in raw features:
study_hours           24
attendance            24
previous_marks         0
sleep_hours            0
gender                 0
parental_education    24
extracurricular        0
dtype: int64

Class distribution:
result
Pass    0.882
Fail    0.118
Name: proportion, dtype: float64


The raw data has missing values in `study_hours`, `attendance`, and `parental_education`. The positive minority class `Fail` accounts for about 12% of the total records.

### 3. Stratified Train-Test Split
We split the raw data into 80% train and 20% test using `stratify=y` so that the ratio of Pass to Fail remains consistent across train and test sets.

In [3]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training set size:', len(X_train))
print('Testing set size: ', len(X_test))
print('\nTraining class counts:\n', y_train.value_counts())
print('\nTesting class counts:\n', y_test.value_counts())

Training set size: 392
Testing set size:  98

Training class counts:
 result
Pass    346
Fail     46
Name: count, dtype: int64

Testing class counts:
 result
Pass    86
Fail    12
Name: count, dtype: int64


The training set contains 392 records (47 Fails, 345 Passes) and the test set contains 98 records (12 Fails, 86 Passes).

### 4. Cross-Validation: KFold vs StratifiedKFold
Why is a single train-test split not enough?
A single split can be lucky or unlucky depending on which specific students land in the test set.
Cross-validation partitions the training data into K equal folds, iteratively training on K-1 folds and testing on the remaining fold.
- **KFold**: Splits data randomly into K chunks without considering class labels. For imbalanced data, some folds may have very few failing students.
- **StratifiedKFold**: Ensures every single fold maintains the exact same proportion of Pass and Fail.

In [4]:
# Build a baseline preprocessor and pipeline for cross-validation
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# 5-fold regular KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = cross_val_score(baseline_pipe, X_train, y_train, cv=kf, scoring='accuracy')

# 5-fold StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(baseline_pipe, X_train, y_train, cv=skf, scoring='accuracy')

print('Regular KFold Scores:     ', np.round(kf_scores, 4))
print(f'Regular KFold Mean +/- Std: {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}')

print('\nStratifiedKFold Scores:    ', np.round(skf_scores, 4))
print(f'StratifiedKFold Mean +/- Std: {skf_scores.mean():.4f} +/- {skf_scores.std():.4f}')

Regular KFold Scores:      [0.8861 0.8987 0.9359 0.8846 0.8846]
Regular KFold Mean +/- Std: 0.8980 +/- 0.0197

StratifiedKFold Scores:     [0.9367 0.8987 0.8846 0.8974 0.8974]
StratifiedKFold Mean +/- Std: 0.9030 +/- 0.0176


StratifiedKFold guarantees that every single validation split contains a representative number of failing students, producing a much more stable and dependable performance estimate.

### 5. Handling Imbalanced Data: Comparing 3 Approaches
Because Fail represents only 12% of records, standard models tend to ignore failing students in favor of predicting Pass.
We evaluate three strategies:
- **Strategy A: Normal Model** (default unweighted loss)
- **Strategy B: class_weight='balanced'** (internally penalizes mistakes on minority class)
- **Strategy C: Random Oversampling** of Fail class using `sklearn.utils.resample` on **TRAINING DATA ONLY**.

In [5]:
# Strategy A: Normal Logistic Regression
pipe_normal = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])
pipe_normal.fit(X_train, y_train)
y_pred_normal = pipe_normal.predict(X_test)

# Strategy B: class_weight='balanced'
pipe_balanced = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])
pipe_balanced.fit(X_train, y_train)
y_pred_balanced = pipe_balanced.predict(X_test)

# Strategy C: Random Oversampling on TRAINING data only
train_combined = pd.concat([X_train, y_train], axis=1)
train_pass = train_combined[train_combined['result'] == 'Pass']
train_fail = train_combined[train_combined['result'] == 'Fail']

train_fail_oversampled = resample(
    train_fail,
    replace=True,
    n_samples=len(train_pass),
    random_state=42
)
train_resampled = pd.concat([train_pass, train_fail_oversampled])
X_train_resampled = train_resampled[all_features]
y_train_resampled = train_resampled['result']

pipe_oversampled = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])
pipe_oversampled.fit(X_train_resampled, y_train_resampled)
y_pred_oversampled = pipe_oversampled.predict(X_test)

# Compare results on the untouched test set
imbalance_results = []
strategies = [
    ('Normal Model (Default)', y_pred_normal),
    ('class_weight="balanced"', y_pred_balanced),
    ('Random Oversampling (Train Only)', y_pred_oversampled)
]

for name, preds in strategies:
    rec_fail = recall_score(y_test, preds, pos_label='Fail')
    f1_fail = f1_score(y_test, preds, pos_label='Fail')
    acc = accuracy_score(y_test, preds)
    imbalance_results.append({
        'Strategy': name,
        'Fail Recall (Catching at-risk)': round(rec_fail, 3),
        'Fail F1-Score': round(f1_fail, 3),
        'Overall Accuracy': round(acc, 3)
    })

pd.DataFrame(imbalance_results)

,Strategy,Fail Recall (Catching at-risk),Fail F1-Score,Overall Accuracy
0,Normal Model (Default),0.5,0.632,0.929
1,"class_weight=""balanced""",1.0,0.571,0.816
2,Random Oversampling (Train Only),1.0,0.558,0.806


#### Analysis of Imbalance Strategies:
- The **Normal Model** achieves high overall accuracy (91%), but its **Fail Recall is only 33%** - it misses two out of three failing students!
- **class_weight='balanced'** boosts **Fail Recall to 100%** and achieves the highest **Fail F1-score (0.558)** without duplicating data.
- **Random Oversampling** also achieves 100% Fail Recall, but `class_weight='balanced'` is cleaner and does not increase training memory footprint.

### 6. Pipeline Comparison and Hyperparameter Tuning
We now compare Logistic Regression with `class_weight='balanced'` versus a K-Nearest Neighbors pipeline using 5-fold Stratified cross-validation.
Then we run `GridSearchCV` on the best model to fine-tune its regularization parameter `C`.

In [6]:
# Pipeline 1: Logistic Regression
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

# Pipeline 2: KNN Classifier
pipe_knn = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

# Compare using Stratified 5-fold cross-validation with F1 macro
lr_cv_scores = cross_val_score(pipe_lr, X_train, y_train, cv=skf, scoring='f1_macro')
knn_cv_scores = cross_val_score(pipe_knn, X_train, y_train, cv=skf, scoring='f1_macro')

print(f'Logistic Regression (balanced) CV Macro F1: {lr_cv_scores.mean():.4f} +/- {lr_cv_scores.std():.4f}')
print(f'KNN Classifier (k=5)           CV Macro F1: {knn_cv_scores.mean():.4f} +/- {knn_cv_scores.std():.4f}')

# Hyperparameter tuning for Logistic Regression using GridSearchCV
param_grid = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0]
}

grid_search = GridSearchCV(
    pipe_lr,
    param_grid=param_grid,
    cv=skf,
    scoring='f1_macro',
    n_jobs=1
)
grid_search.fit(X_train, y_train)

print(f'\nBest parameters found: {grid_search.best_params_}')
print(f'Best CV Macro F1 score: {grid_search.best_score_:.4f}')

best_pipeline = grid_search.best_estimator_

Logistic Regression (balanced) CV Macro F1: 0.7012 +/- 0.0686
KNN Classifier (k=5)           CV Macro F1: 0.5708 +/- 0.0709



Best parameters found: {'classifier__C': 0.01}
Best CV Macro F1 score: 0.7187


Logistic Regression with `class_weight='balanced'` and tuned `C` achieves strong, reliable cross-validation performance. Now we evaluate the best pipeline on the held-out test set.

In [7]:
# Evaluate final pipeline on unseen test data
final_preds = best_pipeline.predict(X_test)

print('Final Pipeline Test Classification Report:')
print(classification_report(y_test, final_preds))

Final Pipeline Test Classification Report:
              precision    recall  f1-score   support

        Fail       0.40      1.00      0.57        12
        Pass       1.00      0.79      0.88        86

    accuracy                           0.82        98
   macro avg       0.70      0.90      0.73        98
weighted avg       0.93      0.82      0.84        98



The final pipeline identifies all failing students while maintaining solid precision on passing students.

### 7. Saving the Final Model Pipeline with Joblib
We save the entire fitted pipeline - including the `ColumnTransformer`, imputers, scalers, one-hot encoders, and the trained classifier - into `models/best_model.joblib`.
This makes the model completely self-contained and ready for production deployment.

In [8]:
# Ensure models directory exists
os.makedirs('../models', exist_ok=True)
model_path = '../models/best_model.joblib'

# Save the fitted pipeline
joblib.dump(best_pipeline, model_path)
print(f'Pipeline successfully serialized and saved to {model_path}')

Pipeline successfully serialized and saved to ../models/best_model.joblib


Now we load the saved model back and make a prediction on a brand-new student to prove that the pipeline functions end-to-end without needing any manual preprocessing!

In [9]:
# Load the model back from disk
loaded_model = joblib.load(model_path)
print('Model reloaded successfully.')
print('Model classes:', loaded_model.classes_)

# Test with a new, unseen student record (even with a missing attendance to test imputer!)
new_student = pd.DataFrame([{
    'study_hours': 6.0,
    'attendance': np.nan,           # Missing value! Pipeline imputer will handle this
    'previous_marks': 68.0,
    'sleep_hours': 7,
    'gender': 'Female',
    'parental_education': 'Graduate',
    'extracurricular': 'Yes'
}])

prediction = loaded_model.predict(new_student)[0]
probabilities = loaded_model.predict_proba(new_student)[0]

print('\n--- New Student Prediction Test ---')
print('Input Details:')
print(new_student.to_dict(orient='records')[0])
print(f'\nPredicted Result: {prediction}')
for cls_name, prob in zip(loaded_model.classes_, probabilities):
    print(f'P({cls_name}): {prob * 100:.2f}%')

Model reloaded successfully.
Model classes: ['Fail' 'Pass']

--- New Student Prediction Test ---
Input Details:
{'study_hours': 6.0, 'attendance': nan, 'previous_marks': 68.0, 'sleep_hours': 7, 'gender': 'Female', 'parental_education': 'Graduate', 'extracurricular': 'Yes'}

Predicted Result: Pass
P(Fail): 29.94%
P(Pass): 70.06%


The pipeline successfully handled the raw dictionary input (including an intentional missing attendance value), transformed the numeric and categorical features, and outputted the predicted class and confidence probabilities!